<a href="https://colab.research.google.com/github/guilhermelaviola/IntelligentCommunication/blob/main/Class08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Integration with Chatbots**
Effective user management is the cornerstone of personalized chatbot experiences, beginning with a robust data model that securely tracks unique IDs, preferences, and interaction history. By implementing profile management features—such as registration, login, and customizable settings—developers can foster user trust while tailoring the bot's language and suggestions to individual needs. Technically, this can be achieved using tools like Python and SQLite3, utilizing encryption and prepared statements to defend against security threats. Ultimately, a successful system must balance this high-level personalization with strict adherence to privacy laws (like LGPD), ensuring transparency in data collection and granting users full control over their personal information.

In [1]:
!pip install PyQt5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 282.4/282.4 kB 6.9 MB/s eta 0:00:00


In [2]:
# Importing all the necessary libraries and resources:
import sys
import nltk
from PyQt5.QtWidgets import QApplication, QMainWindow, QTextEdit, QLineEdit, QPushButton, QVBoxLayout, QWidget
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# NLTK resources:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## **Example: FAQ-style Chatbot**
This code implements a basic FAQ-style Chatbot with a graphical user interface (GUI). Instead of using complex AI, it uses mathematical similarity to match a user's question against a predefined list of jokes/riddles.

In [ ]:
# Defining the data model:
class Question:
  def __init__(self, id, text):
    self.id = id
    self.text = text

class Answer:
  def __init__(self, id, question_id, text):
    self.id = id
    self.question_id = question_id
    self.text = text

# Database for questions and answers:
questions = [
    Question(1, 'What do you call a fake noodle?'),
    Question(2, 'What do you call something that runs but never gets anywhere?'),
    Question(3, 'What do you call something that’s easy to get into, but hard to get out of?')
]

answers = [
    Answer(1, 1, 'An impasta!'),
    Answer(2, 2, 'A refrigerator.'),
    Answer(3, 3, 'Trouble.')
]

# Function to pre process the text:
def preprocess_text(text):
  tokens = word_tokenize(text.lower())
  stop_words = set(stopwords.words('english'))
  filtered_tokens = [token for token in tokens if token not in stop_words and token.isalpha()]
  return ' '.join(filtered_tokens)

# Function to find the best combination:
def find_answer(user_question):
  # Pre processing the questions and answers
  processed_questions = [preprocess_text(p.text) for p in questions]

  # Adding a user question to the question list:
  processed_questions.append(preprocess_text(user_question))

  # Calculating the the similarity using TF-IDF:
  vectorizer = TfidfVectorizer()
  tfidf_matrix = vectorizer.fit_transform(processed_questions)

  # Calculating the similarity between user question and database questions:
  similarity = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1])

  # Finding the most similar question index:
  most_similar_index = similarity.argmax()

  # Returning the correspondent answer:
  most_similar_answer = answers[most_similar_index]
  return most_similar_answer.text

# Defining the user interface using PyQt5:
class ChatbotApp(QMainWindow):
  def __init__(self):
    super().__init__()

    self.setWindowTitle('Chatbot')
    self.setGeometry(100, 100, 500, 400)

    # Interface layout:
    layout = QVBoxLayout()

     # Text box to display the chat:
    self.chat_box = QTextEdit(self)
    self.chat_box.setReadOnly(True)
    layout.addWidget(self.chat_box)

    # Text input for user question:
    self.input_box = QLineEdit(self)
    layout.addWidget(self.input_box)

    # Send button
    self.send_button = QPushButton('Send', self)
    self.send_button.clicked.connect(self.send_question)
    layout.addWidget(self.send_button)

    # Central widget for layout:
    container = QWidget()
    container.setLayout(layout)
    self.setCentralWidget(container)

    # Function to send a question:
    def send_question(self):
      user_question = self.input_box.text()
      if user_question.lower() == 'exit':
        self.close()
      else:
        answer = find_answer(user_question)
        self.chat_box.append('You: ' + user_question)
        self.chat_box.append('Chatbot: ' + answer)
        self.input_box.clear()

# Main function to run the app
def main():
  app = QApplication(sys.argv)
  chatbot = ChatbotApp()
  chatbot.show()
  sys.exit(app.exec_())

if __name__ == '__main__':
    main()
